In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from tqdm.notebook import tqdm
from model.hierarchical_v3 import HierarchicalARM_V3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Load data
cls_test = np.load('../dataset/nsd/embeddings_mindeye2/dinov2_test_sub01.npy')
patches_test = np.load('../dataset/nsd/embeddings_patches/dinov2_patches_test_sub01.npy')[:, 1:, :]
gt_fmri = np.load('../dataset/nsd/mixed_1000_sub01/test_fmri.npy')
print(f"GT fMRI: {gt_fmri.shape}")

In [ ]:
# Load model
model = HierarchicalARM_V3(cls_dim=768, patch_dim=768, num_patches=256, fmri_dim=15724, voxels_per_cluster=30).to(device)
ckpt = torch.load('../checkpoints/hierarchical_v3_subj01_30/best_stage1.pth', map_location=device)
model.load_state_dict(ckpt.get('model_state_dict', ckpt.get('state_dict', ckpt)))
model.eval()
print(f"Model: {model.num_rois} ROIs")

In [ ]:
# Inference
def roi_to_voxel(roi, n_rois, vpc):
    voxel = np.zeros(15724)
    for i in range(n_rois):
        voxel[i*vpc:min((i+1)*vpc, 15724)] = roi[i]
    return voxel

preds = []
with torch.no_grad():
    for i in tqdm(range(0, len(cls_test), 32)):
        pred_roi, _ = model(torch.tensor(cls_test[i:i+32]).to(device), torch.tensor(patches_test[i:i+32]).to(device))
        for roi in pred_roi.cpu().numpy():
            preds.append(roi_to_voxel(roi, model.num_rois, model.voxels_per_cluster))
pred_fmri = np.array(preds)
print(f"Pred: {pred_fmri.shape}")

In [ ]:
# Histogram comparison
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(gt_fmri.flatten(), bins=100, alpha=0.5, density=True, label='GT')
ax.hist(pred_fmri.flatten(), bins=100, alpha=0.5, density=True, label='Pred')
ax.legend()
ax.set_title('Distribution: GT vs Pred')
plt.show()

In [ ]:
# Scatter plot
idx = np.random.choice(len(gt_fmri.flatten()), 20000, replace=False)
plt.figure(figsize=(6,6))
plt.scatter(gt_fmri.flatten()[idx], pred_fmri.flatten()[idx], alpha=0.1, s=1)
plt.plot([-3,3], [-3,3], 'r--')
plt.xlabel('GT'); plt.ylabel('Pred')
plt.title('GT vs Pred')
plt.show()

In [ ]:
# Per-sample correlation
corrs = [stats.pearsonr(gt_fmri[i], pred_fmri[i])[0] for i in range(len(gt_fmri))]
print(f"Correlation: {np.mean(corrs):.4f} ± {np.std(corrs):.4f}")

plt.hist(corrs, bins=30)
plt.axvline(np.mean(corrs), color='r', linestyle='--')
plt.title('Per-sample Correlation')
plt.show()